In [1]:
import tensorflow as tf
import numpy as np

In [2]:
with open("data.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Text length:", len(text))

Text length: 167445


In [3]:
chars = sorted(set(text))

char_to_id = {char: i for i, char in enumerate(chars)}
id_to_char = np.array(chars)

encoded_text = np.array([char_to_id[c] for c in text])

vocab_size = len(chars)

print("Vocabulary size:", vocab_size)

Vocabulary size: 91


In [4]:
SEQ_LENGTH = 100

X = []
y = []

for i in range(len(encoded_text) - SEQ_LENGTH):
    X.append(encoded_text[i:i + SEQ_LENGTH])
    y.append(encoded_text[i + SEQ_LENGTH])

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (167345, 100)
y shape: (167345,)


In [5]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    tf.keras.layers.SimpleRNN(256),

    tf.keras.layers.Dense(vocab_size)
])

model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    )
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
EPOCHS = 20
BATCH_SIZE = 64

model.fit(
    X,
    y,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS
)

Epoch 1/20
2615/2615 ━━━━━━━━━━━━━━━━━━━━ 71s 27ms/step - loss: 2.2723
Epoch 2/20
2615/2615 ━━━━━━━━━━━━━━━━━━━━ 79s 30ms/step - loss: 1.8813
Epoch 3/20
2615/2615 ━━━━━━━━━━━━━━━━━━━━ 224s 86ms/step - loss: 1.7346
Epoch 4/20
2615/2615 ━━━━━━━━━━━━━━━━━━━━ 251s 96ms/step - loss: 1.6430
Epoch 5/20
2615/2615 ━━━━━━━━━━━━━━━━━━━━ 202s 77ms/step - loss: 1.5827
Epoch 6/20
2615/2615 ━━━━━━━━━━━━━━━━━━━━ 164s 62ms/step - loss: 1.5374
Epoch 7/20
2615/2615 ━━━━━━━━━━━━━━━━━━━━ 67s 26ms/step - loss: 1.5016
Epoch 8/20
2615/2615 ━━━━━━━━━━━━━━━━━━━━ 67s 26ms/step - loss: 1.4743
Epoch 9/20
2615/2615 ━━━━━━━━━━━━━━━━━━━━ 67s 26ms/step - loss: 1.4511
Epoch 10/20
2615/2615 ━━━━━━━━━━━━━━━━━━━━ 68s 26ms/step - loss: 1.4319
Epoch 11/20
2615/2615 ━━━━━━━━━━━━━━━━━━━━ 67s 26ms/step - loss: 1.4212
Epoch 12/20
2615/2615 ━━━━━━━━━━━━━━━━━━━━ 68s 26ms/step - loss: 1.4066
Epoch 13/20
2615/2615 ━━━━━━━━━━━━━━━━━━━━ 67s 26ms/step - loss: 1.3973
Epoch 14/20
2615/2615 ━━━━━━━━━━━━━━━━━━━━ 68s 26ms/step - loss: 1.38

In [7]:
def generate_text(seed_text, num_chars=500, temperature=1.0):

    seed_text = seed_text[-SEQ_LENGTH:]

    generated = seed_text

    for _ in range(num_chars):

        input_ids = np.array([
            char_to_id[c] for c in seed_text
        ])

        input_ids = np.expand_dims(input_ids, axis=0)

        logits = model.predict(
            input_ids,
            verbose=0
        )[0]

        logits = logits / temperature

        probabilities = tf.nn.softmax(logits).numpy()

        next_id = np.random.choice(
            vocab_size,
            p=probabilities
        )

        next_char = id_to_char[next_id]

        generated += next_char

        seed_text = seed_text[1:] + next_char

    return generated

In [8]:
seed = text[:SEQ_LENGTH]

generated_text = generate_text(
    seed,
    num_chars=500,
    temperature=0.8
)

print("\nGenerated Text:\n")
print(generated_text)


Generated Text:

The sun was shining brightly in the clear blue sky, and a gentle breeze rustled the leaves of the table...

Ross: We're it if that's gonna ga-listing settival internations.

Changler: Oh, yeah, y'know, Leaves to shared out start from say!

Monica: Alright, you reality in A grabbiry the last seop slapers to the ocayed, the suntlical cornther and steendly from laughten her laked beens.

Heavions medic cance Wattered.

Joey: No. Are you are cultures, and day and eventick, with Characters.]

Chandler: Kid ya and Phoebe: (songer, it's unity and the Amazon raised in our and presses. The sky fonturie
